# Truist Demo - Lakehouse Setup and Synthetic Data\n\nThis notebook creates the required Lakehouse folder structure and Delta tables:\n\n- /Retail/Customers\n- /Retail/Accounts\n- /Retail/Transactions\n- /Commercial/Clients\n- /Commercial/TreasuryPositions\n- /Risk/CreditRiskScores\n- /Risk/EarlyWarningSignals\n- /AML/Alerts\n- /AML/CaseEvents\n- /Digital/EngagementEvents\n\nAll records are synthetic and safe for demo use.

In [ ]:
from pyspark.sql import functions as F\nfrom pyspark.sql.window import Window\n\n# Volume targets for realistic demo scale.\nN_CUSTOMERS = 10000\nN_ACCOUNTS = 25000\nN_TRANSACTIONS = 100000\nN_CLIENTS = 2500\nN_TREASURY_POSITIONS = 15000\nN_EWS = 8000\nN_AML_ALERTS = 5000\nN_CASE_EVENTS = 12000\nN_DIGITAL_EVENTS = 300000\n\nspark.sql(\"SET spark.sql.legacy.timeParserPolicy=LEGACY\")\n\ndef write_delta(path, table_name, df):\n    # Write to a folder path so the physical Lakehouse structure is explicit.\n    df.write.format(\"delta\").mode(\"overwrite\").option(\"overwriteSchema\", \"true\").save(path)\n    spark.sql(f\"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{path}'\")\n    print(f\"{table_name}: {df.count()} rows -> {path}\")\n

In [ ]:
# -----------------------------\n# Retail.Customers\n# -----------------------------\ncustomers = (\n    spark.range(1, N_CUSTOMERS + 1)\n    .withColumn(\"CustomerID\", F.format_string(\"C%06d\", F.col(\"id\")))\n    .withColumn(\n        \"Name\",\n        F.concat(\n            F.element_at(F.array(F.lit(\"Alex\"), F.lit(\"Jordan\"), F.lit(\"Taylor\"), F.lit(\"Morgan\"), F.lit(\"Casey\"), F.lit(\"Avery\"), F.lit(\"Parker\"), F.lit(\"Riley\"), F.lit(\"Quinn\"), F.lit(\"Reese\")), F.pmod(F.col(\"id\"), F.lit(10)) + 1),\n            F.lit(\" \"),\n            F.element_at(F.array(F.lit(\"Carter\"), F.lit(\"Mitchell\"), F.lit(\"Brooks\"), F.lit(\"Sullivan\"), F.lit(\"Hayes\"), F.lit(\"Reynolds\"), F.lit(\"Bennett\"), F.lit(\"Perry\"), F.lit(\"Foster\"), F.lit(\"Cooper\")), F.pmod(F.col(\"id\") * 7, F.lit(10)) + 1)\n        )\n    )\n    .withColumn(\"Segment\", F.when(F.rand(2) < 0.55, F.lit(\"Retail\")).when(F.rand(3) < 0.8, F.lit(\"Affluent\")).otherwise(F.lit(\"Commercial\")))\n    .withColumn(\"DigitalAdoptionScore\", F.round(F.rand(4) * 100, 2))\n    .withColumn(\"RiskTier\", F.when(F.col(\"DigitalAdoptionScore\") < 30, F.lit(\"High\")).when(F.col(\"DigitalAdoptionScore\") < 65, F.lit(\"Medium\")).otherwise(F.lit(\"Low\")))\n    .select(\"CustomerID\", \"Name\", \"Segment\", \"RiskTier\", \"DigitalAdoptionScore\")\n)\n\nwrite_delta(\"Files/Retail/Customers\", \"Retail_Customers\", customers)\n\n# -----------------------------\n# Retail.Accounts\n# -----------------------------\naccounts = (\n    spark.range(1, N_ACCOUNTS + 1)\n    .withColumn(\"AccountID\", F.format_string(\"A%08d\", F.col(\"id\")))\n    .withColumn(\"CustomerID\", F.format_string(\"C%06d\", (F.pmod(F.col(\"id\"), F.lit(N_CUSTOMERS)) + 1).cast(\"int\")))\n    .withColumn(\"ProductType\", F.element_at(F.array(F.lit(\"Checking\"), F.lit(\"Savings\"), F.lit(\"CreditCard\"), F.lit(\"MoneyMarket\"), F.lit(\"CommercialDDA\")), F.pmod(F.col(\"id\"), F.lit(5)) + 1))\n    .withColumn(\"Balance\", F.round(F.rand(5) * 250000 + 500, 2))\n    .withColumn(\"Status\", F.when(F.rand(6) > 0.96, F.lit(\"Closed\")).when(F.rand(7) > 0.90, F.lit(\"Delinquent\")).otherwise(F.lit(\"Active\")))\n    .select(\"AccountID\", \"CustomerID\", \"ProductType\", \"Balance\", \"Status\")\n)\n\nwrite_delta(\"Files/Retail/Accounts\", \"Retail_Accounts\", accounts)\n\n# -----------------------------\n# Retail.Transactions\n# -----------------------------\ntransactions = (\n    spark.range(1, N_TRANSACTIONS + 1)\n    .withColumn(\"TxID\", F.format_string(\"TX%09d\", F.col(\"id\")))\n    .withColumn(\"AccountID\", F.format_string(\"A%08d\", (F.floor(F.rand(8) * N_ACCOUNTS) + 1).cast(\"int\")))\n    .withColumn(\"Amount\", F.round(F.rand(9) * 4500 - 200, 2))\n    .withColumn(\"Channel\", F.element_at(F.array(F.lit(\"Mobile\"), F.lit(\"Online\"), F.lit(\"Branch\"), F.lit(\"ATM\"), F.lit(\"TreasuryPortal\")), F.pmod(F.col(\"id\"), F.lit(5)) + 1))\n    .withColumn(\"Timestamp\", F.from_unixtime(F.unix_timestamp(F.current_timestamp()) - (F.rand(10) * 60 * 24 * 3600).cast(\"int\")).cast(\"timestamp\"))\n    .withColumn(\"MerchantCategory\", F.element_at(F.array(F.lit(\"Grocery\"), F.lit(\"Fuel\"), F.lit(\"Healthcare\"), F.lit(\"Travel\"), F.lit(\"Utilities\"), F.lit(\"ECommerce\"), F.lit(\"Payroll\")), F.pmod(F.col(\"id\"), F.lit(7)) + 1))\n    .select(\"TxID\", \"AccountID\", \"Amount\", \"Channel\", \"Timestamp\", \"MerchantCategory\")\n)\n\nwrite_delta(\"Files/Retail/Transactions\", \"Retail_Transactions\", transactions)\n

In [ ]:
# -----------------------------\n# Commercial.Clients and TreasuryPositions\n# -----------------------------\nclients = (\n    spark.range(1, N_CLIENTS + 1)\n    .withColumn(\"ClientID\", F.format_string(\"CL%06d\", F.col(\"id\")))\n    .withColumn(\"ClientName\", F.concat(F.lit(\"Truist Client \"), F.format_string(\"%05d\", F.col(\"id\"))))\n    .withColumn(\"Industry\", F.element_at(F.array(F.lit(\"Healthcare\"), F.lit(\"Manufacturing\"), F.lit(\"Retail\"), F.lit(\"Technology\"), F.lit(\"Logistics\")), F.pmod(F.col(\"id\"), F.lit(5)) + 1))\n    .withColumn(\"Rating\", F.element_at(F.array(F.lit(\"A\"), F.lit(\"A-\"), F.lit(\"BBB+\"), F.lit(\"BBB\"), F.lit(\"BB+\")), F.pmod(F.col(\"id\"), F.lit(5)) + 1))\n    .select(\"ClientID\", \"ClientName\", \"Industry\", \"Rating\")\n)\nwrite_delta(\"Files/Commercial/Clients\", \"Commercial_Clients\", clients)\n\ntreasury_positions = (\n    spark.range(1, N_TREASURY_POSITIONS + 1)\n    .withColumn(\"PositionID\", F.format_string(\"P%08d\", F.col(\"id\")))\n    .withColumn(\"ClientID\", F.format_string(\"CL%06d\", (F.floor(F.rand(20) * N_CLIENTS) + 1).cast(\"int\")))\n    .withColumn(\"Currency\", F.element_at(F.array(F.lit(\"USD\"), F.lit(\"EUR\"), F.lit(\"GBP\"), F.lit(\"CAD\"), F.lit(\"JPY\")), F.pmod(F.col(\"id\"), F.lit(5)) + 1))\n    .withColumn(\"Amount\", F.round(F.rand(21) * 15000000 + 50000, 2))\n    .withColumn(\"LiquidityBucket\", F.element_at(F.array(F.lit(\"0-7D\"), F.lit(\"8-30D\"), F.lit(\"31-90D\"), F.lit(\"90D+\")), F.pmod(F.col(\"id\"), F.lit(4)) + 1))\n    .select(\"PositionID\", \"ClientID\", \"Currency\", \"Amount\", \"LiquidityBucket\")\n)\nwrite_delta(\"Files/Commercial/TreasuryPositions\", \"Commercial_TreasuryPositions\", treasury_positions)\n

In [ ]:
# -----------------------------\n# Risk and AML domain tables\n# -----------------------------\ncredit_scores = (\n    customers\n    .withColumn(\"Score\", (F.rand(30) * 550 + 300).cast(\"int\"))\n    .withColumn(\"PD\", F.round(F.greatest(F.lit(0.001), (850 - F.col(\"Score\")) / 7000), 4))\n    .withColumn(\"LGD\", F.round(F.least(F.lit(0.85), F.greatest(F.lit(0.15), F.rand(31) * 0.8)), 4))\n    .withColumn(\"UpdatedAt\", F.current_timestamp())\n    .select(\"CustomerID\", \"Score\", \"PD\", \"LGD\", \"UpdatedAt\")\n)\nwrite_delta(\"Files/Risk/CreditRiskScores\", \"Risk_CreditRiskScores\", credit_scores)\n\nearly_warning = (\n    spark.range(1, N_EWS + 1)\n    .withColumn(\"CustomerID\", F.format_string(\"C%06d\", (F.floor(F.rand(32) * N_CUSTOMERS) + 1).cast(\"int\")))\n    .withColumn(\"SignalType\", F.element_at(F.array(F.lit(\"BalanceDrop\"), F.lit(\"MissedPayment\"), F.lit(\"LimitUtilizationSpike\"), F.lit(\"DepositVolatility\")), F.pmod(F.col(\"id\"), F.lit(4)) + 1))\n    .withColumn(\"Severity\", F.element_at(F.array(F.lit(\"Low\"), F.lit(\"Medium\"), F.lit(\"High\")), F.pmod(F.col(\"id\"), F.lit(3)) + 1))\n    .withColumn(\"TriggeredAt\", F.from_unixtime(F.unix_timestamp(F.current_timestamp()) - (F.rand(33) * 14 * 24 * 3600).cast(\"int\")).cast(\"timestamp\"))\n    .select(\"CustomerID\", \"SignalType\", \"Severity\", \"TriggeredAt\")\n)\nwrite_delta(\"Files/Risk/EarlyWarningSignals\", \"Risk_EarlyWarningSignals\", early_warning)\n\naml_alerts = (\n    spark.range(1, N_AML_ALERTS + 1)\n    .withColumn(\"AlertID\", F.format_string(\"AL%08d\", F.col(\"id\")))\n    .withColumn(\"CustomerID\", F.format_string(\"C%06d\", (F.floor(F.rand(34) * N_CUSTOMERS) + 1).cast(\"int\")))\n    .withColumn(\"TxID\", F.format_string(\"TX%09d\", (F.floor(F.rand(35) * N_TRANSACTIONS) + 1).cast(\"int\")))\n    .withColumn(\"RiskScore\", F.round(F.rand(36) * 100, 2))\n    .withColumn(\"Reason\", F.element_at(F.array(F.lit(\"Structuring\"), F.lit(\"Unusual Velocity\"), F.lit(\"High-Risk Geography\"), F.lit(\"Counterparty Concern\")), F.pmod(F.col(\"id\"), F.lit(4)) + 1))\n    .withColumn(\"CreatedAt\", F.from_unixtime(F.unix_timestamp(F.current_timestamp()) - (F.rand(37) * 10 * 24 * 3600).cast(\"int\")).cast(\"timestamp\"))\n    .select(\"AlertID\", \"CustomerID\", \"TxID\", \"RiskScore\", \"Reason\", \"CreatedAt\")\n)\nwrite_delta(\"Files/AML/Alerts\", \"AML_Alerts\", aml_alerts)\n\ncase_events = (\n    spark.range(1, N_CASE_EVENTS + 1)\n    .withColumn(\"CaseEventID\", F.format_string(\"CE%09d\", F.col(\"id\")))\n    .withColumn(\"AlertID\", F.format_string(\"AL%08d\", (F.floor(F.rand(38) * N_AML_ALERTS) + 1).cast(\"int\")))\n    .withColumn(\"EventType\", F.element_at(F.array(F.lit(\"Opened\"), F.lit(\"Reviewed\"), F.lit(\"Escalated\"), F.lit(\"Closed\")), F.pmod(F.col(\"id\"), F.lit(4)) + 1))\n    .withColumn(\"Owner\", F.concat(F.lit(\"Investigator_\"), F.format_string(\"%03d\", (F.pmod(F.col(\"id\"), F.lit(120)) + 1).cast(\"int\"))))\n    .withColumn(\"EventTimestamp\", F.from_unixtime(F.unix_timestamp(F.current_timestamp()) - (F.rand(39) * 20 * 24 * 3600).cast(\"int\")).cast(\"timestamp\"))\n    .select(\"CaseEventID\", \"AlertID\", \"EventType\", \"Owner\", \"EventTimestamp\")\n)\nwrite_delta(\"Files/AML/CaseEvents\", \"AML_CaseEvents\", case_events)\n

In [ ]:
# -----------------------------\n# Digital.EngagementEvents\n# -----------------------------\ndigital_events = (\n    spark.range(1, N_DIGITAL_EVENTS + 1)\n    .withColumn(\"CustomerID\", F.format_string(\"C%06d\", (F.floor(F.rand(50) * N_CUSTOMERS) + 1).cast(\"int\")))\n    .withColumn(\"EventType\", F.element_at(F.array(F.lit(\"Login\"), F.lit(\"BillPay\"), F.lit(\"Transfer\"), F.lit(\"CardControls\"), F.lit(\"TreasuryApproval\"), F.lit(\"ProfileUpdate\")), F.pmod(F.col(\"id\"), F.lit(6)) + 1))\n    .withColumn(\"Device\", F.when(F.rand(51) > 0.62, F.lit(\"Mobile\")).when(F.rand(52) > 0.5, F.lit(\"Web\")).otherwise(F.lit(\"Tablet\")))\n    .withColumn(\"Timestamp\", F.from_unixtime(F.unix_timestamp(F.current_timestamp()) - (F.rand(53) * 30 * 24 * 3600).cast(\"int\")).cast(\"timestamp\"))\n    .select(\"CustomerID\", \"EventType\", \"Device\", \"Timestamp\")\n)\nwrite_delta(\"Files/Digital/EngagementEvents\", \"Digital_EngagementEvents\", digital_events)\n\nprint(\"Lakehouse setup and synthetic data generation complete.\")\n

## Copilot prompts (semantic model)\n\n-- Explain why Customer 123 moved into high-risk tier.\n-- Summarize AML alerts for the last 24 hours.\n-- Describe liquidity changes across buckets.\n-- Generate a client-ready narrative for digital engagement trends.